# Image Predictor

This notebook loads the CNN model you've trained (`cnn_classifier_model.h5`) and allows you to predict the category of any eye image. Simply provide the path to the image you want to analyze.

In [1]:
# Import Required Libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Check TensorFlow version and GPU availability
print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.experimental.list_physical_devices('GPU')) > 0)

TensorFlow version: 2.10.1
GPU Available: True


## Load the Trained Model

First, we'll load the saved model that you've already trained.

In [ ]:
# Configuration
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 5
MODEL_PATH = 'your model path'

# Class labels for 5-class classification
CLASS_LABELS = {
    0: 'No DR (Normal)',
    1: 'Mild DR',
    2: 'Moderate DR',
    3: 'Severe DR',
    4: 'Proliferative DR'
}

# Binary classification: Class 0 is Normal, Classes 1-4 are Abnormal (DR detected)
BINARY_LABELS = {
    0: 'Normal (No DR)',
    1: 'Abnormal (DR Detected)'
}

# Load the trained model
try:
    model = load_model(MODEL_PATH)
    print(f"✓ Model successfully loaded from: {MODEL_PATH}")
except Exception as e:
    print(f"Error loading model: {e}")

✓ Model successfully loaded from: Models\cnn_classifier_model2.h5


In [3]:
# Display model summary
print("\nModel Architecture:")
model.summary()

# Display number of parameters
total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")


Model Architecture:
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_24 (Conv2D)          (None, 224, 224, 32)      896       
                                                                 
 batch_normalization_28 (Bat  (None, 224, 224, 32)     128       
 chNormalization)                                                
                                                                 
 conv2d_25 (Conv2D)          (None, 224, 224, 32)      9248      
                                                                 
 batch_normalization_29 (Bat  (None, 224, 224, 32)     128       
 chNormalization)                                                
                                                                 
 max_pooling2d_10 (MaxPoolin  (None, 112, 112, 32)     0         
 g2D)                                                            
                                 

## Define Prediction Function

We'll create a function that takes an image path, preprocesses the image, and returns predictions.

In [ ]:
def predict_image(image_path, model, img_height=224, img_width=224):
    """
    Predict the class of an input image with both 5-class and binary classification
    
    Args:
        image_path: Path to the image file
        model: Trained Keras model
        img_height: Target height for resizing
        img_width: Target width for resizing
    
    Returns:
        predicted_class: Predicted class (0-4)
        confidence: Confidence score for 5-class
        all_probabilities: Probabilities for all 5 classes
        binary_class: Binary classification (0=Normal, 1=Abnormal)
        binary_confidence: Confidence score for binary classification
        img: Loaded image
    """
    try:
        # Load and preprocess the image
        img = load_img(image_path, target_size=(img_height, img_width))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
        img_array = img_array / 255.0  # Normalize
        
        # Make prediction (5-class)
        predictions = model.predict(img_array, verbose=0)
        predicted_class = np.argmax(predictions[0])
        confidence = np.max(predictions[0])
        
        # Binary classification
        # Class 0 = Normal, Classes 1-4 = Abnormal (DR detected)
        normal_prob = predictions[0][0]  # Probability of class 0 (No DR)
        abnormal_prob = np.sum(predictions[0][1:])  # Sum of probabilities for classes 1-4
        
        binary_class = 0 if normal_prob > abnormal_prob else 1
        binary_confidence = normal_prob if binary_class == 0 else abnormal_prob
        
        return predicted_class, confidence, predictions[0], binary_class, binary_confidence, img
    
    except Exception as e:
        print(f"Error predicting image {image_path}: {e}")
        return None, None, None, None, None, None

## Create User Interface for Image Input

Now let's create a simple interface for users to input image paths or select from sample images.

In [ ]:
def display_prediction_results(image_path):
    """Display the prediction results for an image with both 5-class and binary classification"""
    
    # Make prediction
    predicted_class, confidence, probabilities, binary_class, binary_confidence, img = predict_image(
        image_path, model, IMG_HEIGHT, IMG_WIDTH
    )
    
    if predicted_class is None:
        print("Prediction failed. Please check the image path.")
        return
    
    # Display the results
    print(f"\n{'='*60}")
    print(f"PREDICTION RESULTS")
    print(f"{'='*60}")
    print(f"Image: {os.path.basename(image_path)}")
    
    # Binary Classification Results
    print(f"\n--- BINARY CLASSIFICATION ---")
    print(f"Result: {BINARY_LABELS[binary_class]}")
    print(f"Confidence: {binary_confidence:.4f} ({binary_confidence*100:.2f}%)")
    
    # 5-Class Classification Results
    print(f"\n--- DETAILED 5-CLASS CLASSIFICATION ---")
    print(f"Predicted Class: {predicted_class} - {CLASS_LABELS[predicted_class]}")
    print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
    
    print("\nAll class probabilities:")
    for i, prob in enumerate(probabilities):
        print(f"  Class {i} ({CLASS_LABELS[i]}): {prob:.4f} ({prob*100:.2f}%)")
    
    # Create visualization with 3 subplots
    plt.figure(figsize=(18, 5))
    
    # Plot 1: The image
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"Input Image", fontsize=12, fontweight='bold')
    plt.axis('off')
    
    # Plot 2: Binary classification
    plt.subplot(1, 3, 2)
    binary_probs = [probabilities[0], np.sum(probabilities[1:])]
    colors_binary = ['green' if binary_class == 0 else 'lightgray', 
                     'red' if binary_class == 1 else 'lightgray']
    bars_binary = plt.bar(['Normal', 'Abnormal (DR)'], [p * 100 for p in binary_probs], color=colors_binary)
    plt.ylabel('Probability (%)', fontsize=10)
    plt.title(f'Binary Classification\n{BINARY_LABELS[binary_class]} ({binary_confidence*100:.2f}%)', 
              fontsize=12, fontweight='bold')
    plt.ylim(0, 100)
    
    # Add percentage labels on bars
    for bar, prob in zip(bars_binary, binary_probs):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{prob*100:.1f}%', ha='center', va='bottom', fontsize=9)
    
    # Plot 3: 5-Class classification probabilities
    plt.subplot(1, 3, 3)
    colors_5class = ['green' if i == predicted_class else 'lightcoral' if i > 0 else 'lightgray' 
                     for i in range(NUM_CLASSES)]
    bars = plt.bar(range(NUM_CLASSES), probabilities * 100, color=colors_5class)
    plt.xticks(range(NUM_CLASSES), [f'Class {i}' for i in range(NUM_CLASSES)], rotation=45, ha='right')
    plt.xlabel('Class', fontsize=10)
    plt.ylabel('Probability (%)', fontsize=10)
    plt.title(f'5-Class Classification\nClass {predicted_class}: {CLASS_LABELS[predicted_class]} ({confidence*100:.2f}%)', 
              fontsize=12, fontweight='bold')
    plt.ylim(0, 100)
    
    # Add percentage labels on bars
    for bar, prob in zip(bars, probabilities):
        height = bar.get_height()
        if height > 5:  # Only show label if bar is tall enough
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f'{prob*100:.1f}%', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Summary recommendation
    print(f"\n{'='*60}")
    if binary_class == 0:
        print("✓ SUMMARY: No diabetic retinopathy detected.")
    else:
        severity = CLASS_LABELS[predicted_class]
        print(f"⚠ SUMMARY: Diabetic retinopathy detected - {severity}")
        print("  Recommendation: Consult with an ophthalmologist for proper diagnosis.")
    print(f"{'='*60}\n")

In [6]:
# Create a simple function to handle user input
def predict_from_path():
    """Function to predict from a user-provided image path"""
    
    print("Enter the path to the image file:")
    print("(Example: 'sampleimages\\eye18.png')")
    
    # Get user input
    image_path = input("Image path: ")
    
    # Check if the file exists
    if not os.path.exists(image_path):
        print(f"Error: File '{image_path}' not found!")
        return
    
    # Display prediction results
    display_prediction_results(image_path)

## Advanced UI with Widgets

Let's create a more interactive interface using IPython widgets. This allows for a more user-friendly experience.

In [7]:
def create_prediction_ui():
    """Create an interactive UI for image prediction"""
    
    # Create widgets
    path_input = widgets.Text(
        value='sampleimages\\eye18.png',
        placeholder='Enter image path',
        description='Image Path:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    predict_button = widgets.Button(
        description='Predict',
        button_style='primary',
        tooltip='Click to predict'
    )
    
    output = widgets.Output()
    
    # Define button click handler
    def on_predict_button_clicked(b):
        with output:
            clear_output()
            if not os.path.exists(path_input.value):
                print(f"Error: File '{path_input.value}' not found!")
                return
            display_prediction_results(path_input.value)
    
    predict_button.on_click(on_predict_button_clicked)
    
    # Display the widgets
    display(widgets.VBox([
        widgets.HTML(value="<h3>Enter the path to an image:</h3>"),
        path_input,
        predict_button,
        output
    ]))

# Create the UI
create_prediction_ui()

## Test with Sample Images

Let's test our predictor with some sample images to demonstrate how it works.

In [8]:
# Test with a sample image
sample_image_path = 'M:\Code\Minor project\DR V2 DS2\sample\sample\15_right.jpeg'

# Check if the sample image exists
if os.path.exists(sample_image_path):
    print(f"Testing with sample image: {sample_image_path}")
    display_prediction_results(sample_image_path)
else:
    print(f"Sample image not found at {sample_image_path}")
    print("You can run the prediction on your own images using the UI above.")

_right.jpege not found at M:\Code\Minor project\DR V2 DS2\sample\sample
You can run the prediction on your own images using the UI above.


## Batch Prediction

If you have multiple images to predict, you can use this function to process them all at once.

In [ ]:
def batch_predict(image_folder, image_extension='.png'):
    """Predict classes for all images in a folder with both binary and 5-class classification"""
    
    if not os.path.exists(image_folder):
        print(f"Error: Folder '{image_folder}' not found!")
        return
    
    # Get all images with the specified extension
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(image_extension)]
    
    if not image_files:
        print(f"No {image_extension} files found in {image_folder}")
        return
    
    print(f"Found {len(image_files)} images. Processing...")
    
    # Create a table of results
    results = []
    
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        predicted_class, confidence, _, binary_class, binary_confidence, _ = predict_image(img_path, model)
        
        if predicted_class is not None:
            results.append({
                'Image': img_file,
                'Binary Class': BINARY_LABELS[binary_class],
                'Binary Confidence': binary_confidence,
                'Detailed Class': f"{predicted_class} - {CLASS_LABELS[predicted_class]}",
                'Detailed Confidence': confidence
            })
    
    # Display results
    print("\n" + "="*120)
    print("BATCH PREDICTION RESULTS")
    print("="*120)
    print(f"{'Image':<30} {'Binary':<20} {'Conf':<8} {'Detailed Class':<30} {'Conf':<8}")
    print("-" * 120)
    
    for r in results:
        print(f"{r['Image']:<30} {r['Binary Class']:<20} {r['Binary Confidence']*100:>6.2f}% "
              f"{r['Detailed Class']:<30} {r['Detailed Confidence']*100:>6.2f}%")
    
    # Count predictions by binary class
    print("\n--- BINARY CLASSIFICATION SUMMARY ---")
    binary_counts = {'Normal (No DR)': 0, 'Abnormal (DR Detected)': 0}
    for r in results:
        binary_counts[r['Binary Class']] += 1
    
    for label, count in binary_counts.items():
        percentage = (count / len(results)) * 100
        print(f"{label}: {count} images ({percentage:.1f}%)")
    
    # Count predictions by 5-class
    print("\n--- DETAILED 5-CLASS DISTRIBUTION ---")
    class_counts = {}
    for r in results:
        class_label = r['Detailed Class']
        if class_label in class_counts:
            class_counts[class_label] += 1
        else:
            class_counts[class_label] = 1
    
    for class_label, count in sorted(class_counts.items()):
        percentage = (count / len(results)) * 100
        print(f"{class_label}: {count} images ({percentage:.1f}%)")
    
    print("="*120)

In [10]:
# Example of batch prediction (uncomment to use)
# To use this, replace 'your_image_folder' with the path to your folder of images
# batch_predict('sampleimages')

## Instructions for Use

1. **Single Image Prediction**:
   - Use the interactive UI above to enter the path to your image
   - Click the "Predict" button to see the results

2. **Batch Prediction**:
   - Uncomment the last cell and replace 'sampleimages' with your folder path
   - Run the cell to predict classes for all images in the folder

3. **Understanding Results**:
   - **Binary Classification**: Classifies as Normal (No DR) or Abnormal (DR Detected)
     - Class 0 = Normal (No Diabetic Retinopathy)
     - Class 1 = Abnormal (Any stage of DR detected)
   - **5-Class Classification**: Provides detailed severity grading
     - Class 0: No DR (Normal)
     - Class 1: Mild DR
     - Class 2: Moderate DR
     - Class 3: Severe DR
     - Class 4: Proliferative DR
   - The confidence scores indicate how certain the model is about each prediction
   - The visualizations show probabilities for both classification types

In [11]:
print("\n" + "="*50)
print("IMAGE PREDICTOR READY")
print("="*50)
print("You can now use this notebook to predict the category of any eye image.")
print("Just enter the path to your image in the UI above or use the batch prediction function.")
print("\nHappy predicting!")


IMAGE PREDICTOR READY
You can now use this notebook to predict the category of any eye image.
Just enter the path to your image in the UI above or use the batch prediction function.

Happy predicting!
